# Network Data Exploration & Quality Analysis

Explores the **same training snapshots** used in `test_training_and_serving_alvaro.ipynb`.
Loads directly from `notebook_artifacts/snapshots/` (the pkl files written by step_ingest).
Falls back to a live Spanner fetch using identical arguments if the cache is absent.

## Sections
1. Setup & Configuration
2. Load Snapshots
3. Snapshot Summary
4. Build DataFrames (all GNN features)
5. Aggregate Statistics
6. Per-Node Statistics
7. Feature Distributions
8. Temporal Evolution
9. Data Quality
10. Topology Analysis
11. Topology Visualisation

## 1. Setup & Configuration

In [10]:
import os
import sys
import pickle
import argparse
import datetime
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from IPython.display import display

src_path = str((Path.cwd().resolve().parent / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from utils.gnn_utils import (
    ROUTER_FEATURES, INTERFACE_FEATURES, BGP_SESSION_FEATURES, FEATURE_DIMS,
)
import run_pipeline_local

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded.")

Libraries loaded.


In [11]:
# ── Credentials — identical to test_training_and_serving_alvaro.ipynb ─────────
_local_creds = str((Path.cwd().resolve().parent / "src" / "networkagent.json").resolve())
creds_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", _local_creds)
if creds_path and os.path.exists(creds_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_path
    print(f"Using service-account credentials: {creds_path}")
else:
    print("networkagent.json not found — will use Application Default Credentials (ADC)")

project_id       = os.getenv("GOOGLE_CLOUD_PROJECT", "agents-1234")
spanner_instance = os.getenv("SPANNER_INSTANCE", "networktopology-instance")
spanner_database = os.getenv("SPANNER_DATABASE", "networktopology-db")
interval_minutes = 0.5   # 30-second cadence — matches training notebook

# ── Same time windows as the training notebook ────────────────────────────────
TRAIN_FROM = datetime.datetime(2026, 4, 14, 18,  0, 0)
TRAIN_TO   = datetime.datetime(2026, 4, 14, 19, 45, 0)

def _snapshots_in_window(t_from, t_to):
    return int((t_to - t_from).total_seconds() / (interval_minutes * 60))

train_num = _snapshots_in_window(TRAIN_FROM, TRAIN_TO)

train_args = argparse.Namespace(
    project=project_id,
    spanner_instance=spanner_instance,
    spanner_database=spanner_database,
    interval_minutes=interval_minutes,
    num_snapshots=train_num,
    from_time=TRAIN_FROM,
    to_time=TRAIN_TO,
)

output_dir    = Path("notebook_artifacts")
snapshots_dir = output_dir / "snapshots"

print(f"Project         : {project_id}")
print(f"Instance        : {spanner_instance}/{spanner_database}")
print(f"Training window : {TRAIN_FROM.isoformat()} → {TRAIN_TO.isoformat()}")
print(f"                  ({train_num} snapshots @ {interval_minutes*60:.0f}s interval)")

Using service-account credentials: /Users/alvarossl/Documents/bt_gnn/gnnsandbox/gnn/src/networkagent.json
Project         : agents-1234
Instance        : networktopology-instance/networktopology-db
Training window : 2026-04-14T18:00:00 → 2026-04-14T19:45:00
                  (210 snapshots @ 30s interval)


## 2. Load Snapshots

Loads from `notebook_artifacts/snapshots/*.pkl` — the same files written by the training notebook.
Falls back to a live Spanner fetch (using identical arguments) if the cache is absent.

In [12]:
snapshots = []
pkl_files = sorted(snapshots_dir.glob("*.pkl")) if snapshots_dir.exists() else []

if pkl_files:
    for p in pkl_files:
        with open(p, "rb") as f:
            snapshots.append(pickle.load(f))
    print(f"Loaded {len(snapshots)} snapshots from disk  ({snapshots_dir})")
else:
    print(f"No cached snapshots at {snapshots_dir} — fetching from Spanner...")
    snapshots = run_pipeline_local.step_ingest(train_args, snapshots_dir)
    print(f"Fetched and saved {len(snapshots)} snapshots.")

if snapshots:
    print(f"Time range : {snapshots[0]['timestamp']}  →  {snapshots[-1]['timestamp']}")

Loaded 211 snapshots from disk  (notebook_artifacts/snapshots)
Time range : 2026-04-14T18:00:00  →  2026-04-14T19:45:00


## 3. Snapshot Summary

In [13]:
# ── Per-snapshot summary table (identical to training notebook) ───────────────
_hdr = (f"{'#':>4}  {'Timestamp':<26}  {'Routers':>9}  "
        f"{'Interfaces':>12}  {'BGPSessions':>13}  {'Edges':>7}")
print(f"\n{_hdr}")
print("-" * len(_hdr))
for _i, _s in enumerate(snapshots):
    _r   = [n for n in _s['nodes'] if n['type'] == 'router']
    _ifc = [n for n in _s['nodes'] if n['type'] == 'interface']
    _bgp = [n for n in _s['nodes'] if n['type'] == 'bgp_session']
    _r_up  = sum(1 for n in _r   if n.get('state') == 1.0)
    _i_up  = sum(1 for n in _ifc if n.get('state') == 1.0)
    _b_up  = sum(1 for n in _bgp if n.get('bgp_state') == 1.0)
    _ec = {}
    for _e in _s['edges']:
        _ec[_e['relation']] = _ec.get(_e['relation'], 0) + 1
    _etag = '  '.join(f"{k}={v}" for k, v in sorted(_ec.items()))
    print(
        f"{_i+1:>4}  {_s['timestamp']:<26}  "
        f"{_r_up}/{len(_r):<2} up  "
        f"{_i_up}/{len(_ifc):<4} up  "
        f"{_b_up}/{len(_bgp):<3} estab  "
        f"{len(_s['edges']):>7}  ({_etag})"
    )

# ── High-level counts ─────────────────────────────────────────────────────────
if snapshots:
    _last = snapshots[-1]
    _ifc_nodes = [n for n in _last['nodes'] if n['type'] == 'interface']
    _bgp_nodes = [n for n in _last['nodes'] if n['type'] == 'bgp_session']
    _grads = [abs(n.get('rx_err_gradient', 0.0)) for n in _ifc_nodes]
    print(f"\n── Temporal features (last snapshot) ──────────────────────")
    print(f"  rx_err_gradient    max={max(_grads) if _grads else 0:.6f}")
    _deltas = [n.get('prefix_count_delta', 0.0) for n in _bgp_nodes]
    print(f"  prefix_count_delta sessions with delta≠0: {sum(1 for d in _deltas if d!=0)}/{len(_deltas)}")


   #  Timestamp                     Routers    Interfaces    BGPSessions    Edges
---------------------------------------------------------------------------------
   1  2026-04-14T18:00:00         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has_interface=66  interface_of=66  ospf_peer=182)
   2  2026-04-14T18:00:30         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has_interface=66  interface_of=66  ospf_peer=182)
   3  2026-04-14T18:01:00         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has_interface=66  interface_of=66  ospf_peer=182)
   4  2026-04-14T18:01:30         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has_interface=66  interface_of=66  ospf_peer=182)
   5  2026-04-14T18:02:00         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has_interface=66  interface_of=66  ospf_peer=182)
   6  2026-04-14T18:02:30         14/14 up  52/66   up  0/0   estab      530  (connected_to=216  has

## 4. Build DataFrames

Extracts **all GNN features** for each node type as defined in `gnn_utils.py`.

In [ ]:
router_rows, iface_rows, bgp_rows = [], [], []

# Build a hostname lookup from all snapshots (in case a router appears only in some)
_router_id_to_hostname = {}

for snap in snapshots:
    ts = snap['timestamp']
    for node in snap['nodes']:
        t = node['type']
        if t == 'router':
            _router_id_to_hostname[node['id']] = node.get('hostname', node['id'])
            router_rows.append({
                'timestamp':       ts,
                'id':              node['id'],
                'hostname':        node.get('hostname', ''),
                'role':            node.get('role', ''),
                # GNN features
                'state':           node.get('state',           0.0),
                'cpu':             node.get('cpu',             0.0),
                'mem':             node.get('mem',             0.0),
                'ospf_num_routes': node.get('ospf_num_routes', 0.0),
                'pfx_count_norm':  node.get('pfx_count_norm',  0.0),
            })
        elif t == 'interface':
            iface_rows.append({
                'timestamp':       ts,
                'id':              node['id'],
                'name':            node.get('name',           ''),
                'device_id':       node.get('device_id',      ''),
                # GNN features
                'state':           node.get('state',           0.0),
                'rx_drops':        node.get('rx_drops',        0.0),
                'tx_drops':        node.get('tx_drops',        0.0),
                'mtu_norm':        node.get('mtu_norm',        0.0),
                'rx_errs_rate':    node.get('rx_errs_rate',    0.0),
                'rx_bytes_rate':   node.get('rx_bytes_rate',   0.0),
                'tx_bytes_rate':   node.get('tx_bytes_rate',   0.0),
                'rx_err_gradient': node.get('rx_err_gradient', 0.0),
                'tx_util':         node.get('tx_util',         0.0),
                'rx_util':         node.get('rx_util',         0.0),
            })
        elif t == 'bgp_session':
            bgp_rows.append({
                'timestamp':           ts,
                'id':                  node['id'],
                'router_id':           node.get('router_id',           ''),
                'peer_ip':             node.get('peer_ip',             ''),
                'bgp_state':           node.get('bgp_state',           0.0),
                'pfx_count_raw':       node.get('pfx_count_raw',       0.0),
                'prefix_count_delta':  node.get('prefix_count_delta',  0.0),
                'session_uptime_norm': node.get('session_uptime_norm', 0.0),
            })

df_r = pd.DataFrame(router_rows)
df_i = pd.DataFrame(iface_rows)
df_b = pd.DataFrame(bgp_rows)

for df in [df_r, df_i, df_b]:
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])

# Resolve router hostname for interface rows
if not df_i.empty:
    df_i['router_name'] = df_i['device_id'].map(_router_id_to_hostname).fillna(df_i['device_id'])

ROUTER_METRICS    = ['state','cpu','mem','ospf_num_routes','pfx_count_norm']
INTERFACE_METRICS = ['state','rx_drops','tx_drops','mtu_norm','rx_errs_rate',
                     'rx_bytes_rate','tx_bytes_rate','rx_err_gradient','tx_util','rx_util']
BGP_METRICS       = ['bgp_state','pfx_count_raw','prefix_count_delta','session_uptime_norm']

print("DataFrames built:")
print(f"  Routers      : {len(df_r):>7,} rows  ({df_r['id'].nunique() if not df_r.empty else 0} unique nodes)")
print(f"  Interfaces   : {len(df_i):>7,} rows  ({df_i['id'].nunique() if not df_i.empty else 0} unique nodes)")
print(f"  BGP Sessions : {len(df_b):>7,} rows  ({df_b['id'].nunique() if not df_b.empty else 0} unique nodes)")

## 5. Aggregate Statistics

Descriptive statistics across all snapshots and all nodes, for every GNN feature.

In [ ]:
def _describe(df, metrics, label):
    if df.empty:
        print(f"{label}: no data"); return
    print(f"\n{'='*65}")
    print(f"{label} — Aggregate Statistics  (N={len(df):,} rows, {df['id'].nunique()} nodes)")
    print(f"{'='*65}")
    desc = df[metrics].describe().T
    desc = desc.rename(columns={'50%': 'median'})
    display(desc)

_describe(df_r, ROUTER_METRICS,    "ROUTER")
_describe(df_i, INTERFACE_METRICS, "INTERFACE")
_describe(df_b, BGP_METRICS,       "BGP SESSION")

## 6. Per-Node Statistics

Mean, std, min, and max for each metric **per individual node** across all snapshots it appeared in.

In [ ]:
# ── Routers: one row per hostname ─────────────────────────────────────────────
if not df_r.empty:
    print("── ROUTER: per-node statistics ─────────────────────────────────────────────")
    _r_grp = df_r.groupby('hostname')[ROUTER_METRICS].agg(['mean','std','min','max'])
    _r_grp.columns = ['_'.join(c) for c in _r_grp.columns]
    display(_r_grp.round(4))

# ── Interfaces: top 20 by mean tx_util ───────────────────────────────────────
if not df_i.empty:
    print("\n── INTERFACE: per-node statistics (top 20 by mean tx_util) ─────────────────")
    _i_stats = (
        df_i.groupby(['id','name','router_name'])[INTERFACE_METRICS]
        .mean()
        .sort_values('tx_util', ascending=False)
    )
    display(_i_stats.head(20).round(4))

    print("\n── INTERFACE: per-node statistics (top 20 by mean rx_drops) ────────────────")
    display(
        df_i.groupby(['id','name','router_name'])[INTERFACE_METRICS]
        .mean()
        .sort_values('rx_drops', ascending=False)
        .head(20)
        .round(4)
    )

In [ ]:
# ── Heatmap: mean metric value per router node ────────────────────────────────
if not df_r.empty:
    heat_r = df_r.groupby('hostname')[ROUTER_METRICS].mean()
    fig, ax = plt.subplots(figsize=(10, max(4, len(heat_r) * 0.5)))
    sns.heatmap(heat_r, annot=True, fmt='.3f', cmap='Blues', ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Mean value'})
    ax.set_title("Router: mean feature value per node (across all snapshots)")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Hostname")
    plt.tight_layout()
    plt.show()

if not df_i.empty:
    heat_i = df_i.groupby('router_name')[INTERFACE_METRICS].mean()
    fig, ax = plt.subplots(figsize=(14, max(4, len(heat_i) * 0.5)))
    sns.heatmap(heat_i, annot=True, fmt='.4f', cmap='Oranges', ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Mean value (avg across ifaces per router)'})
    ax.set_title("Interface: mean feature value per router (averaged across interfaces)")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Router")
    plt.tight_layout()
    plt.show()

## 7. Feature Distributions

Histogram of every GNN feature across all nodes and all snapshots.

In [ ]:
def _plot_distributions(df, metrics, title, color):
    if df.empty: return
    ncols = 3
    nrows = (len(metrics) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = axes.flatten()
    for ax, feat in zip(axes, metrics):
        vals = df[feat].dropna()
        ax.hist(vals, bins=50, color=color, edgecolor='none', alpha=0.8)
        ax.axvline(vals.mean(), color='crimson', linestyle='--', linewidth=1.5,
                   label=f'mean={vals.mean():.4f}')
        ax.axvline(vals.median(), color='navy', linestyle=':', linewidth=1.2,
                   label=f'median={vals.median():.4f}')
        ax.set_title(feat, fontsize=10)
        ax.set_xlabel("Value", fontsize=8)
        ax.set_ylabel("Count", fontsize=8)
        ax.legend(fontsize=6)
        ax.tick_params(labelsize=7)
    for ax in axes[len(metrics):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

_plot_distributions(df_r, ROUTER_METRICS,    "Router Feature Distributions (all snapshots)",    "steelblue")
_plot_distributions(df_i, INTERFACE_METRICS, "Interface Feature Distributions (all snapshots)", "darkorange")
if not df_b.empty:
    _plot_distributions(df_b, BGP_METRICS, "BGP Session Feature Distributions", "seagreen")

## 8. Temporal Evolution

How key metrics change over the training window for each node.

In [ ]:
# ── Router time series ────────────────────────────────────────────────────────
if not df_r.empty:
    fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

    for hostname, grp in df_r.groupby('hostname'):
        grp = grp.sort_values('timestamp')
        axes[0].plot(grp['timestamp'], grp['cpu'], label=hostname, linewidth=0.8, alpha=0.85)
    axes[0].set_title("Router CPU (node_load1)")
    axes[0].set_ylabel("Load")
    axes[0].legend(fontsize=6, ncol=4, loc='upper right')
    axes[0].grid(True, alpha=0.3)

    for hostname, grp in df_r.groupby('hostname'):
        grp = grp.sort_values('timestamp')
        axes[1].plot(grp['timestamp'], grp['mem'], label=hostname, linewidth=0.8, alpha=0.85)
    axes[1].set_title("Router Memory (normalised to 4 GB)")
    axes[1].set_ylabel("Fraction [0, 1]")
    axes[1].grid(True, alpha=0.3)

    # Down events: scatter hostnames on y-axis where state=0
    df_r_down = df_r[df_r['state'] == 0.0]
    hostnames_ordered = sorted(df_r['hostname'].unique())
    y_map = {h: i for i, h in enumerate(hostnames_ordered)}
    if not df_r_down.empty:
        axes[2].scatter(
            df_r_down['timestamp'],
            df_r_down['hostname'].map(y_map),
            s=15, color='red', zorder=3, marker='x'
        )
    axes[2].set_yticks(range(len(hostnames_ordered)))
    axes[2].set_yticklabels(hostnames_ordered, fontsize=7)
    axes[2].set_title("Router Down Events (state = 0)")
    axes[2].set_xlabel("Time")
    axes[2].grid(True, alpha=0.3)
    if df_r_down.empty:
        axes[2].text(0.5, 0.5, 'No down events in this window',
                     transform=axes[2].transAxes, ha='center', va='center', color='gray')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Interface time series ─────────────────────────────────────────────────────
if not df_i.empty:
    fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

    # TX/RX util: plot mean + max band across all interfaces
    util_max  = df_i.groupby('timestamp')[['tx_util','rx_util']].max()
    util_mean = df_i.groupby('timestamp')[['tx_util','rx_util']].mean()
    axes[0].fill_between(util_max.index, util_mean['tx_util'], util_max['tx_util'],
                         alpha=0.2, color='steelblue')
    axes[0].plot(util_mean.index, util_mean['tx_util'], color='steelblue',
                 linewidth=1, label='Mean TX util')
    axes[0].plot(util_max.index,  util_max['tx_util'],  color='steelblue',
                 linewidth=0.5, linestyle='--', label='Max TX util')
    axes[0].fill_between(util_max.index, util_mean['rx_util'], util_max['rx_util'],
                         alpha=0.2, color='darkorange')
    axes[0].plot(util_mean.index, util_mean['rx_util'], color='darkorange',
                 linewidth=1, label='Mean RX util')
    axes[0].set_title("Interface Utilisation (mean and max across all interfaces)")
    axes[0].set_ylabel("Utilisation [0, 1]")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Drops: total per timestamp
    drop_ts = df_i.groupby('timestamp')[['rx_drops','tx_drops']].sum()
    axes[1].plot(drop_ts.index, drop_ts['rx_drops'], label='Total RX drops', linewidth=1)
    axes[1].plot(drop_ts.index, drop_ts['tx_drops'], label='Total TX drops',
                 linewidth=1, linestyle='--')
    axes[1].set_title("Total Interface Drops (sum across all interfaces)")
    axes[1].set_ylabel("Drops/s")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # RX error rate: max across interfaces
    err_ts = df_i.groupby('timestamp')[['rx_errs_rate','rx_err_gradient']].max()
    axes[2].plot(err_ts.index, err_ts['rx_errs_rate'],    label='Max rx_errs_rate', linewidth=1)
    axes[2].plot(err_ts.index, err_ts['rx_err_gradient'], label='Max rx_err_gradient',
                 linewidth=1, linestyle='--')
    axes[2].set_title("Interface Error Rate & Gradient (max across all interfaces)")
    axes[2].set_xlabel("Time")
    axes[2].set_ylabel("Rate")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Interface down events ─────────────────────────────────────────────────────
if not df_i.empty:
    df_i_down = df_i[df_i['state'] == 0.0]
    down_count = df_i_down.groupby('id').size().sort_values(ascending=False)
    if not down_count.empty:
        print(f"Interfaces with state=0 events (top 20):")
        display(down_count.head(20).rename('down_snapshots').to_frame())

        # Timeline heatmap: which interfaces were down when
        # Sample timestamps to keep the heatmap readable
        _ifaces_with_downs = down_count.head(20).index.tolist()
        _ts_sample = sorted(df_i['timestamp'].unique())[::max(1, len(df_i['timestamp'].unique())//40)]
        _heat_df = (
            df_i[df_i['id'].isin(_ifaces_with_downs)]
            .pivot_table(index='id', columns='timestamp', values='state', aggfunc='min')
        )
        # Shorten ID labels
        _heat_df.index = _heat_df.index.str.split(':').str[-1]
        _heat_df.columns = [str(c)[:16] for c in _heat_df.columns]

        fig, ax = plt.subplots(figsize=(16, max(4, len(_heat_df) * 0.4)))
        sns.heatmap(_heat_df, cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
                    cbar_kws={'label': 'state (1=up, 0=down)'}, linewidths=0)
        ax.set_title("Interface State Timeline (top 20 most-down interfaces)")
        ax.set_xlabel("Timestamp")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=6)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
        plt.tight_layout()
        plt.show()
    else:
        print("No interface down events detected in the training window.")

## 9. Data Quality

For each feature: zero rate, null rate, constant flag, coefficient of variation, and unique value count.  
Features with >95% zeros or std≈0 are flagged — they contribute nothing to the GNN and may indicate a missing Prometheus metric.

In [ ]:
def data_quality_report(df, metrics, label):
    if df.empty:
        print(f"{label}: no data"); return
    data = df[metrics]
    report = pd.DataFrame(index=metrics)
    report['zero_rate_%']   = (data == 0).mean() * 100
    report['null_rate_%']   = data.isnull().mean() * 100
    report['mean']          = data.mean()
    report['std']           = data.std()
    report['cv']            = (data.std() / data.mean().abs()).replace([np.inf, np.nan], np.nan)
    report['min']           = data.min()
    report['p25']           = data.quantile(0.25)
    report['median']        = data.median()
    report['p75']           = data.quantile(0.75)
    report['max']           = data.max()
    report['n_unique']      = data.nunique()
    report['constant']      = data.std() < 1e-9
    print(f"\n{'='*72}")
    print(f"{label} — Data Quality  (N={len(data):,} rows, {df['id'].nunique()} unique nodes)")
    print(f"{'='*72}")
    display(report.round(4))
    const = report[report['constant']].index.tolist()
    zero_dom = report[report['zero_rate_%'] > 95].index.tolist()
    if const:
        print(f"  ⚠  Constant features (std < 1e-9):        {const}")
    if zero_dom:
        print(f"  ⚠  Zero-dominated features (>95% zeros):  {zero_dom}")
    low_var = report[(report['cv'].fillna(0) < 0.01) & ~report['constant']].index.tolist()
    if low_var:
        print(f"  ⚠  Near-constant features (CV < 0.01):    {low_var}")

data_quality_report(df_r, ROUTER_METRICS,    "ROUTER")
data_quality_report(df_i, INTERFACE_METRICS, "INTERFACE")
if not df_b.empty:
    data_quality_report(df_b, BGP_METRICS, "BGP SESSION")

In [ ]:
# ── Zero-rate heatmaps ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(4, len(_router_id_to_hostname) * 0.55)))

if not df_r.empty:
    zero_r = df_r.groupby('hostname')[ROUTER_METRICS].apply(lambda g: (g == 0).mean() * 100)
    sns.heatmap(zero_r, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
                linewidths=0.5, vmin=0, vmax=100,
                cbar_kws={'label': '% zero values'})
    axes[0].set_title("Router: zero rate (%) per feature per node")
    axes[0].set_xlabel("Feature")
    axes[0].set_ylabel("Hostname")
    axes[0].tick_params(axis='x', rotation=45)

if not df_i.empty:
    zero_i = df_i.groupby('router_name')[INTERFACE_METRICS].apply(lambda g: (g == 0).mean() * 100)
    sns.heatmap(zero_i, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1],
                linewidths=0.5, vmin=0, vmax=100,
                cbar_kws={'label': '% zero values'})
    axes[1].set_title("Interface: zero rate (%) per feature per router\n(averaged across all interfaces of that router)")
    axes[1].set_xlabel("Feature")
    axes[1].set_ylabel("Router")
    axes[1].tick_params(axis='x', rotation=45)

plt.suptitle("Data Quality: Zero Rate Heatmaps", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Node presence across snapshots (completeness) ─────────────────────────────
n_snaps = len(snapshots)

print("── Router presence across snapshots ────────────────────────────────────────")
if not df_r.empty:
    router_presence = df_r.groupby('hostname')['timestamp'].nunique().sort_values()
    router_presence_pct = (router_presence / n_snaps * 100).round(1)
    display(pd.DataFrame({'snapshots_present': router_presence,
                          'pct_of_total': router_presence_pct}))

print("\n── Interface presence across snapshots (bottom 20) ─────────────────────────")
if not df_i.empty:
    iface_presence = df_i.groupby('id')['timestamp'].nunique().sort_values()
    iface_pct = (iface_presence / n_snaps * 100).round(1)
    display(pd.DataFrame({'snapshots_present': iface_presence,
                          'pct_of_total': iface_pct}).head(20))

## 10. Topology Analysis

Graph structure of the latest snapshot: degree distribution, role breakdown, interface and OSPF peer counts per router.

In [ ]:
latest = snapshots[-1]

# Build directed graph
G = nx.DiGraph()
_node_meta = {}
for node in latest['nodes']:
    G.add_node(node['id'],
               ntype=node['type'],
               hostname=node.get('hostname', node.get('name', node['id'])),
               role=node.get('role', ''),
               state=node.get('state', node.get('bgp_state', 1.0)))
    _node_meta[node['id']] = G.nodes[node['id']]

for edge in latest['edges']:
    if edge['source'] in G and edge['target'] in G:
        G.add_edge(edge['source'], edge['target'], relation=edge['relation'])

_iface_to_router = {n['id']: n['device_id']
                    for n in latest['nodes'] if n['type'] == 'interface'}

# ── Role breakdown ─────────────────────────────────────────────────────────────
role_counts = defaultdict(int)
for n, d in G.nodes(data=True):
    if d['ntype'] == 'router':
        role_counts[d['role']] += 1
print("Router roles:", dict(role_counts))

# ── Interface count per router ─────────────────────────────────────────────────
ifaces_per_router = defaultdict(int)
for iid, rid in _iface_to_router.items():
    ifaces_per_router[rid] += 1

print("\nInterface count per router:")
for rid, cnt in sorted(ifaces_per_router.items(), key=lambda x: -x[1]):
    hn = _node_meta.get(rid, {}).get('hostname', rid)
    print(f"  {hn:<22} {cnt:>3} interfaces")

# ── OSPF peer count per router ────────────────────────────────────────────────
ospf_degree = defaultdict(int)
for u, v, data in G.edges(data=True):
    if data['relation'] == 'ospf_peer':
        ospf_degree[u] += 1

print("\nOSPF peer count per router:")
for rid, cnt in sorted(ospf_degree.items(), key=lambda x: -x[1]):
    hn = _node_meta.get(rid, {}).get('hostname', rid)
    print(f"  {hn:<22} {cnt:>3} OSPF peers")

# ── Edge type summary ─────────────────────────────────────────────────────────
edge_type_counts = defaultdict(int)
for u, v, data in G.edges(data=True):
    edge_type_counts[data['relation']] += 1
print("\nEdge counts by relation:")
for rel, cnt in sorted(edge_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {rel:<25} {cnt:>5}")

In [ ]:
# ── Degree distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Router OSPF degree
router_ospf_deg = [ospf_degree.get(n, 0)
                   for n, d in G.nodes(data=True) if d['ntype'] == 'router']
axes[0].hist(router_ospf_deg, bins=range(max(router_ospf_deg or [0])+2),
             color='steelblue', edgecolor='white', rwidth=0.8)
axes[0].set_title("Router OSPF Peer Degree")
axes[0].set_xlabel("# OSPF peers")
axes[0].set_ylabel("# Routers")

# Interface: # physical connections (connected_to degree)
iface_conn_deg = defaultdict(int)
for u, v, data in G.edges(data=True):
    if data['relation'] == 'connected_to':
        iface_conn_deg[u] += 1
iface_deg_vals = [iface_conn_deg.get(n, 0)
                  for n, d in G.nodes(data=True) if d['ntype'] == 'interface']
axes[1].hist(iface_deg_vals, bins=range(max(iface_deg_vals or [0])+2),
             color='darkorange', edgecolor='white', rwidth=0.8)
axes[1].set_title("Interface Physical Connections")
axes[1].set_xlabel("# connected_to edges")
axes[1].set_ylabel("# Interfaces")

# Role pie chart
roles = list(role_counts.keys())
counts = list(role_counts.values())
role_colors_pie = {'P':'#888888','PE':'#4C9BE8','RR':'#E85C4C','CE':'#5CB85C'}
axes[2].pie(counts, labels=roles, autopct='%1.0f%%',
            colors=[role_colors_pie.get(r, '#dddddd') for r in roles],
            startangle=90)
axes[2].set_title("Router Role Distribution")

plt.suptitle("Topology Degree Analysis", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Topology Visualisation

**Top plot**: Router OSPF graph. Node size ∝ OSPF degree. Node colour = role. Faded = state=0.  
**Bottom plot**: Full router–interface bipartite layout. Routers top band, interfaces bottom band.

In [ ]:
ROLE_COLOR = {'P':'#888888', 'PE':'#4C9BE8', 'RR':'#E85C4C', 'CE':'#5CB85C'}

# ── Router OSPF graph ─────────────────────────────────────────────────────────
R = nx.Graph()
for node in latest['nodes']:
    if node['type'] == 'router':
        R.add_node(node['id'],
                   hostname=node.get('hostname', ''),
                   role=node.get('role', ''),
                   state=node.get('state', 1.0))

for edge in latest['edges']:
    u, v = edge['source'], edge['target']
    if edge['relation'] == 'ospf_peer' and u in R and v in R:
        R.add_edge(u, v)

node_colors_r = [ROLE_COLOR.get(R.nodes[n]['role'], '#dddddd') for n in R.nodes()]
node_alpha_r  = [1.0 if R.nodes[n]['state'] == 1.0 else 0.25 for n in R.nodes()]
node_sizes_r  = [500 + 120 * R.degree(n) for n in R.nodes()]
node_labels_r = {n: R.nodes[n]['hostname'] for n in R.nodes()}

pos_r = nx.spring_layout(R, seed=42, k=1.8)

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(R, pos_r, ax=ax, alpha=0.35, width=1.5, edge_color='gray')
for i, n in enumerate(R.nodes()):
    nx.draw_networkx_nodes(R, pos_r, nodelist=[n], ax=ax,
                           node_color=[node_colors_r[i]],
                           node_size=[node_sizes_r[i]],
                           alpha=node_alpha_r[i])
nx.draw_networkx_labels(R, pos_r, labels=node_labels_r, ax=ax,
                        font_size=8, font_weight='bold')

legend_handles = [
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor=c, markersize=13, label=role)
    for role, c in ROLE_COLOR.items()
]
legend_handles.append(
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor='gray', alpha=0.25, markersize=13, label='state=0 (down)')
)
ax.legend(handles=legend_handles, title='Role', loc='upper left', fontsize=9)
ax.set_title(
    f"Router OSPF Topology — {latest['timestamp']}\n"
    f"(node size ∝ OSPF degree, faded = state=0)",
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Router–Interface bipartite view ───────────────────────────────────────────
H = nx.Graph()
for node in latest['nodes']:
    if node['type'] in ('router', 'interface'):
        H.add_node(node['id'],
                   ntype=node['type'],
                   label=node.get('hostname', node.get('name', '')),
                   role=node.get('role', ''))

for edge in latest['edges']:
    u, v = edge['source'], edge['target']
    if edge['relation'] in ('has_interface', 'connected_to') and u in H and v in H:
        H.add_edge(u, v, relation=edge['relation'])

routers_h = [n for n, d in H.nodes(data=True) if d['ntype'] == 'router']
ifaces_h  = [n for n, d in H.nodes(data=True) if d['ntype'] == 'interface']

# Band layout: routers evenly spaced at y=2, interfaces at y=0
pos_h = {}
for i, n in enumerate(sorted(routers_h,
                              key=lambda n: H.nodes[n]['label'])):
    pos_h[n] = (i * 2.5, 2.0)
for i, n in enumerate(sorted(ifaces_h)):
    pos_h[n] = (i * 0.38, 0.0)

r_colors_h = [ROLE_COLOR.get(H.nodes[n]['role'], '#888888') for n in routers_h]
edge_colors_h = [
    'lightgray' if H.edges[e]['relation'] == 'has_interface' else 'steelblue'
    for e in H.edges()
]

fig, ax = plt.subplots(figsize=(20, 7))
nx.draw_networkx_nodes(H, pos_h, nodelist=routers_h, node_color=r_colors_h,
                       node_size=900, ax=ax)
nx.draw_networkx_nodes(H, pos_h, nodelist=ifaces_h, node_color='#B3D9F7',
                       node_size=60, ax=ax, alpha=0.85)
nx.draw_networkx_edges(H, pos_h, ax=ax, edge_color=edge_colors_h,
                       alpha=0.35, width=0.6)
nx.draw_networkx_labels(H, pos_h,
                        labels={n: H.nodes[n]['label'] for n in routers_h},
                        ax=ax, font_size=7, font_weight='bold')
ax.set_title(
    "Router–Interface Topology\n"
    "(routers top band, interfaces bottom band; "
    "grey edges = has_interface, blue = connected_to)",
    fontsize=11
)
ax.axis('off')
plt.tight_layout()
plt.show()